In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [2]:
words = open('names.txt', 'r').read().splitlines()
words[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

In [3]:
len(words)

32033

In [4]:
chars = sorted(list(set(''.join(words))))
stoi = {c: i+1 for i, c in enumerate(chars)}
stoi['.'] = 0
itos = {i: c for c, i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(stoi)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
{'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'x': 24, 'y': 25, 'z': 26, '.': 0}


In [5]:
# build the datasets
block_size = 3
def build_dataset(words):
    X, Y = [], []

    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [6]:
def cmp(s, dt, t):
    ex = torch.all(dt == t.grad).item()
    app = torch.allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

In [9]:
W1.shape

torch.Size([30, 64])

In [10]:
# MLP revisited

n_emb = 10 # the dim of the char embedding vectors
n_hidden = 64 # the number of neurons in the hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647)
C = torch.rand((vocab_size, n_emb), generator=g) # (27, 10)
# layer 1
W1 = torch.randn((n_emb * block_size, n_hidden), generator=g)   * (5/3)/((n_emb * block_size)**0.5)
b1 = torch.randn(n_hidden, generator=g)                         * 0.1 # can be left out because of bnbias
# layer 2
W2 = torch.randn((n_hidden, vocab_size), generator=g)           * 0.1
b2 = torch.randn(vocab_size, generator=g)                       * 0.1
# batchnorm parameters
bngain = torch.ones((1, n_hidden)) # batch normalization gain
bnbias = torch.zeros((1, n_hidden)) # batch normalization bias
bnmean_running = torch.zeros((1, n_hidden))
bnstd_running = torch.ones((1, n_hidden))

parameters = [C, W1, b1, W2, b2, bnbias, bngain]
print(sum(p.nelement() for p in parameters)) # num of params in total
for p in parameters:
    p.requires_grad = True

4137


In [13]:
Xtr[ix].shape

torch.Size([32, 3])

In [ ]:
batch_size = 32

# construct a minibatch
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g) # batch_size = 32 random int values between 0 and 182625
Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y, 32 * random 3 + 1 training + gt-values

In [85]:
Xtr.shape

torch.Size([182625, 3])

In [ ]:
# forward pass, "chunkated" into smaller steps that are possible to backward one at a time

n = batch_size # a shorter variable also, for convenience

emb = C[Xb] # embed the characters into vectors, (32, 3, 10)
embcat = emb.view(emb.shape[0], -1) # concatenate the vectors, (32, 30)

# Linear layer 1
hprebn = embcat @ W1 + b1 # hidden layer pre-activation (32, 30) @ (30, 64) + (1, 64) -->  (32, 64)

# BatchNorm layer
bnmeani = 1/n*hprebn.sum(0, keepdim=True) # 1/n * (1, 64) --> (1, 64)
bndiff = hprebn - bnmeani # (32, 64) with bnmean broadcasted over all values of x in hprebn
bndiff2 = bndiff**2 # (32, 64) with all values raised to the power of 2
bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n-1, not n), HUOM! otosvarianssi, 1/(n-1) * (1, 64)
bnvar_inv = (bnvar + 1e-5)**-0.5 # note the use of epsilon value 1e-5, 1 / (1, 64)
bnraw = bndiff * bnvar_inv # (32, 64) * 1 / sqrt(1, 64) with bnvar_inv broadcasted over all values of x in bndiff -> (32, 64)
hpreact = bngain * bnraw + bnbias # scale and shift, (32, 64)

# Non-linearity
h = torch.tanh(hpreact) # hidden layer, (32, 64)

# Linear layer 2
logits = h @ W2 + b2 # output layer, (32, 64) @ (64, 27) --> (32, 27)

# cross entropy loss (same as F.cross_entropy(logits, Yb))
logit_maxes = logits.max(1, keepdim=True).values # picks the max value per batch row, (32, 1)
norm_logits = logits - logit_maxes # subtract max for numerical stability, (32, 27) with logit_maxes broadcasted over all values of x in logits
counts = norm_logits.exp() # (32, 27)
counts_sum = counts.sum(1, keepdims=True) # (32, 1)
counts_sum_inv = counts_sum**-1 # (32, 1), if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
probs = counts * counts_sum_inv # (32, 27), probability by x / total
logprobs = probs.log() # (32, 27), log of probabilities
loss = -logprobs[range(n), Yb].mean() # picks the index of the correct Yb values probabilities from -logprobs and calculates the 
                                      # mean of these values, higher loss -> lower probability assigned to the correct answer

# PyTorch backward pass
for p in parameters:
  p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv,
          norm_logits, logit_maxes, logits, h, hpreact, bnraw,
         bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani,
         embcat, emb]:
  t.retain_grad()
loss.backward()
loss

tensor(3.4357, grad_fn=<NegBackward0>)

In [125]:
counts.shape

torch.Size([32, 27])

In [ ]:
# Exercise 1: backprop through the whole thing manually, 
# backpropagating through exactly all of the variables 
# as they are defined in the forward pass above, one by one
dlogprobs = torch.zeros_like(logprobs) # initiate the tensor
dlogprobs[range(n), Yb] = -1.0/n # only correct positions get non-zero values
dprobs = dlogprobs * 1 / probs
dcounts_sum_inv = (dprobs * counts).sum(1, keepdim=True)    # counts_sum_inv is broadcasted, so it must be summed in the same dim as its broadcasted forward
                                                            # i.e. upon broadcasting, counts_sum_inv is used 27 times, so they all need to be collected
dcounts = counts_sum_inv * dprobs
dcounts_sum = dcounts_sum_inv * (-counts_sum**-2)
dcounts += dcounts_sum
dnorm_logits = norm_logits.exp() * dcounts

cmp('logprobs', dlogprobs, logprobs)
cmp('probs', dprobs, probs)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)
cmp('counts_sum', dcounts_sum, counts_sum)
cmp('counts', dcounts, counts)
cmp('norm_logits', dnorm_logits, norm_logits)
# cmp('logit_maxes', dlogit_maxes, logit_maxes)
# cmp('logits', dlogits, logits)
# cmp('h', dh, h)
# cmp('W2', dW2, W2)
# cmp('b2', db2, b2)
# cmp('hpreact', dhpreact, hpreact)
# cmp('bngain', dbngain, bngain)
# cmp('bnbias', dbnbias, bnbias)
# cmp('bnraw', dbnraw, bnraw)
# cmp('bnvar_inv', dbnvar_inv, bnvar_inv)
# cmp('bnvar', dbnvar, bnvar)
# cmp('bndiff2', dbndiff2, bndiff2)
# cmp('bndiff', dbndiff, bndiff)
# cmp('bnmeani', dbnmeani, bnmeani)
# cmp('hprebn', dhprebn, hprebn)
# cmp('embcat', dembcat, embcat)
# cmp('W1', dW1, W1)
# cmp('b1', db1, b1)
# cmp('emb', demb, emb)
# cmp('C', dC, C)

logprobs        | exact: True  | approximate: True  | maxdiff: 0.0
probs           | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0
counts          | exact: True  | approximate: True  | maxdiff: 0.0
norm_logits     | exact: True  | approximate: True  | maxdiff: 0.0


In [133]:
dlogprobs = 0

dprobs = 0
dcounts_sum_inv = 0
dcounts = 0
dcounts_sum = 0
dcounts = 0
dnorm_logits = 0
dlogits = 0
dlogit_maxes = 0

In [ ]:
# dlogprobs = torch.zeros_like(logprobs)
# dlogprobs[range(n), Yb] = -1.0/n
# dprobs = dlogprobs * (1 / probs)
# dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True)
# dcounts = counts_sum_inv * dprobs
# dcounts_sum = (-counts_sum**-2) * dcounts_sum_inv
# dcounts += torch.ones_like(counts) * dcounts_sum # different branch, gradients add
# dnorm_logits = norm_logits.exp() * dcounts
# dlogits = dnorm_logits.clone()
# dlogit_maxes = (-dnorm_logits).sum(1, keepdim=True)